# Preprocessing — Phase 2 — Curry 9 (.cdt)

**Instructions:**
1. Select the **paths** (.cdt folder, JSON config, output folder; `quality_summary.tsv` optional but recommended)
2. Adjust the **preprocessing parameters** and **rejection thresholds**
3. Click **Load participants** to list participants and their channels
4. For each participant: adjust the **channels** to include, then check the participants to process
5. Click **Run preprocessing**

**Output structure:**

```
{output_folder}/
  reports_preprocessing/
    {id}_preprocessing_report.html    ← HTML report with artifact heatmap
    {id}_epoch_rejection.tsv          ← 1 row/epoch: file_id, epoch_idx, stage, reject_flag, flag per method
    {id}_rejection_summary.tsv        ← rejection statistics per stage and per method (per participant)
    global_epoch_rejection.tsv        ← concatenation of all participants' epoch_rejection files
    global_rejection_by_stage.tsv     ← pivot table: 1 row/stage, n and % rejected total + per method
  derivatives/
    [group sub-folders mirroring the .cdt folder structure, if any]
    {id}_all-epo.fif                  ← all epochs + rejection metadata (MNE)
```

**Note:** Epochs are NOT removed here. Phase 2b will allow inspection of rejected epochs and validation/correction of the mask before saving a final `_clean-epo.fif`.

In [ ]:
import os
import io
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import mne
import yasa

import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
from specparam import SpectralModel
import sys as _sys
# curry shared modules — located next to this notebook
_here = os.path.dirname(os.path.abspath('__file__'))
if _here not in _sys.path:
    _sys.path.insert(0, _here)
from curry_header import read_curry_header
from curry_io import load_events_curry, rec_start_from_header

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')


In [ ]:
# ---- Rejection-method registry (single source of truth) ----
# 'event' is the optional 6th method (epochs containing selected scored events). The helpers
# below iterate METHOD_ORDER so 'event' is handled when present and silently ignored when absent.
METHOD_ORDER  = ['amplitude', 'flat', 'gradient', '1f_error', '1f_r2', 'event']
METHOD_CODE   = {m: i + 1 for i, m in enumerate(METHOD_ORDER)}   # 1..6  (0 = none)
MULTIPLE_CODE = len(METHOD_ORDER) + 1                            # 7 = multiple
# Human-readable method names for report table headers (keyed by METHOD_ORDER labels).
METHOD_LABEL  = {'amplitude': 'Amplitude', 'flat': 'Flat', 'gradient': 'Gradient',
                 '1f_error': '1/f error', '1f_r2': '1/f R²', 'event': 'Event'}


# --- Custom (non-AASM) sleep stages -------------------------------------------
# Declared in config_param/custom_stages.json (written by 3_remap_hypno) and recognised here
# so per-stage rejection thresholds, summaries and the heatmap include them.
BASE_STAGE_COLORS = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
# Distinct, non-red palette for custom stages (red is reserved for REM on the hypnogram).
CUSTOM_STAGE_PALETTE = ['#8dd3c7', '#ffffb3', '#bebada', '#80b1d3', '#fdb462', '#b3de69', '#fccde5', '#d9d9d9']


def load_custom_stages(folder):
    """Read config_param/custom_stages.json; return list of kept non-AASM labels ([] if absent/unreadable)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def parse_custom_field(text):
    """Parse the comma-separated 'Custom stages' field into a clean, de-duplicated list."""
    seen = []
    for tok in str(text).split(','):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.append(tok)
    return seen


def custom_stage_style(custom_stages):
    """Stage -> y-position, stage -> colour, and axis ticks for AASM + custom stages.
    AASM keeps YASA heights (W top ... N3 bottom); custom stages stack below N3 (-1, -2, ...).
    Returns (stage_y, stage_colors, ytick_pos, ytick_labels)."""
    stage_y = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    stage_colors = dict(BASE_STAGE_COLORS)
    for i, cs in enumerate(custom_stages):
        stage_y[cs] = -1 - i
        stage_colors[cs] = CUSTOM_STAGE_PALETTE[i % len(CUSTOM_STAGE_PALETTE)]
    ordered = sorted(stage_y.items(), key=lambda kv: kv[1])
    ytick_pos = [y for _, y in ordered]
    ytick_labels = ['REM' if s == 'R' else s for s, _ in ordered]
    return stage_y, stage_colors, ytick_pos, ytick_labels


# ---- Event loading: Curry French text export (*_ScoredEvents_Export.txt) ----
# Wrapper matching the EDF loader's (path, suffix) -> (DataFrame, source) signature so the
# downstream call sites stay unchanged. It reads the .cdt.dpo header for the recording-start
# datetime, then curry_io.load_events_curry converts the export's clock times to seconds.
def load_events(cdt_path, event_suffix='_ScoredEvents_Export.txt'):
    """Load Curry scored events next to the .cdt file. Returns
    (DataFrame Name/Start/Duration in seconds, 'txt') or (None, None)."""
    cdt_path = Path(cdt_path)
    ev_path = cdt_path.with_name(f'{cdt_path.stem}{event_suffix}')
    if not ev_path.exists():
        return None, None
    try:
        hdr = read_curry_header(str(cdt_path))
        rec_start = rec_start_from_header(hdr)
        if rec_start is None:
            return None, None
        df = load_events_curry(str(ev_path), rec_start)
        if df is None or df.empty:
            return None, None
        return df, 'txt'
    except Exception:
        return None, None


def load_event_remap(path):
    """Lenient loader for event_remap.json (strict parse, then repair one trailing comma
    before a closing } or ]). Returns {raw_label: canonical_label_or_None}."""
    with open(path, encoding='utf-8') as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        import re
        return json.loads(re.sub(r',(\s*[}\]])', r'\1', text))


def compute_event_epoch_mask(events_df, event_remap, selected_types, n_epochs,
                             epoch_sec=30.0):
    """Flag each 30 s epoch that CONTAINS the onset of any selected canonical event type.

    events_df      : DataFrame Name/Start/Duration (seconds) from load_events, or None
    event_remap    : {raw_label: canonical_label_or_None} from event_remap.json
    selected_types : set/list of canonical labels chosen by the user
    Returns a (n_epochs,) bool array (epoch-level; replicated across channels later).
    Containment rule: an epoch is flagged when an event's ONSET (Start) falls inside it.
    The annotated Duration is intentionally IGNORED -- clinicians often score only the
    event onset, not a precise duration, so the duration field is not trusted to widen
    the rejection. Each matching event therefore flags exactly one 30 s epoch.
    """
    mask = np.zeros(n_epochs, dtype=bool)
    selected = set(selected_types)
    if events_df is None or not selected:
        return mask
    for _, ev in events_df.iterrows():
        canonical = event_remap.get(str(ev['Name']))
        if canonical is None or canonical not in selected:
            continue
        start = float(ev['Start'])
        if not np.isfinite(start) or start < 0:
            continue
        e = int(np.floor(start / epoch_sec))
        if 0 <= e < n_epochs:
            mask[e] = True
    return mask


def compute_rejection_masks(epochs, sfreq, hypno_epochs, ptp_thresholds,
                             flat_thresh, grad_thresh,
                             psd_freqs, psds_data_uV2, thresh_error, thresh_r2,
                             event_epoch_mask=None, fit_fmin=2.0, fit_fmax=45.0, progress=None):
    """
    Compute per-(epoch, channel) boolean rejection masks for each method.

    Curry per-channel variant (memory): reads the signal ONE CHANNEL AT A TIME from `epochs`
    (epochs.get_data(picks=[ci])) so a high-density montage is never materialised as a second
    full (n_epochs, n_channels, n_times) float64 array. The formulas are identical to the EDF
    full-array version, so the per-(epoch, channel) masks are byte-identical.

    progress(done, total, msg) is called periodically for a UI progress bar (per channel for the
    time-domain pass, then per epoch for the 1/f fits).

    Returns dict of (n_epochs, n_channels) bool arrays:
        keys: 'amplitude', 'flat', 'gradient', '1f_error', '1f_r2' (+ 'event' if provided)
    """
    n_epochs = len(epochs)
    n_channels = len(epochs.ch_names)

    ptp_thresh_arr = np.array([ptp_thresholds.get(s, 250.0) for s in hypno_epochs])  # (n_epochs,)

    mask_amplitude = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_flat      = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_gradient  = np.zeros((n_epochs, n_channels), dtype=bool)

    do_1f = psds_data_uV2 is not None and psd_freqs is not None
    total = n_channels + (n_epochs if do_1f else 0)
    done = 0

    # Time-domain methods, one channel at a time (amplitude / flat / gradient).
    for ci in range(n_channels):
        ch = epochs.get_data(picks=[ci])[:, 0, :] * 1e6   # (n_epochs, n_times) in uV
        ptp = np.ptp(ch, axis=-1)
        mask_amplitude[:, ci] = ptp > ptp_thresh_arr
        mask_flat[:, ci]      = ptp < flat_thresh
        mask_gradient[:, ci]  = np.max(np.abs(np.diff(ch, axis=-1)), axis=-1) > grad_thresh
        del ch
        done += 1
        if progress is not None:
            progress(done, total, 'scanning channels')

    # 1/f fit quality via specparam (per epoch x channel) — same model as the EDF version.
    mask_1f_error = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_1f_r2    = np.zeros((n_epochs, n_channels), dtype=bool)
    if do_1f:
        freq_mask = (psd_freqs >= fit_fmin) & (psd_freqs <= fit_fmax)
        freqs_reduced = psd_freqs[freq_mask]
        sp_model = SpectralModel(
            peak_width_limits=[0.5, 20],
            aperiodic_mode='fixed',
            min_peak_height=0.3,
        )
        for ei in range(n_epochs):
            for ci in range(n_channels):
                psd_reduced = psds_data_uV2[ei, ci, freq_mask]
                try:
                    sp_model.fit(freqs_reduced, psd_reduced)
                    error = sp_model.get_metrics('error', 'mae')
                    r2    = sp_model.get_metrics('gof', 'squared')
                    if error > thresh_error:
                        mask_1f_error[ei, ci] = True
                    if r2 < thresh_r2:
                        mask_1f_r2[ei, ci] = True
                except Exception:
                    mask_1f_error[ei, ci] = True
                    mask_1f_r2[ei, ci]    = True
            done += 1
            if progress is not None and (ei % 25 == 0 or ei == n_epochs - 1):
                progress(done, total, '1/f fit')

    mask_dict = {
        'amplitude': mask_amplitude,
        'flat':      mask_flat,
        'gradient':  mask_gradient,
        '1f_error':  mask_1f_error,
        '1f_r2':     mask_1f_r2,
    }
    if event_epoch_mask is not None:
        mask_dict['event'] = np.tile(np.asarray(event_epoch_mask, dtype=bool)[:, np.newaxis],
                                     (1, n_channels))
    return mask_dict


def build_combined_method_matrix(mask_dict):
    """
    Build an (n_epochs, n_channels) integer matrix encoding which method first
    flagged each cell.  0=none, then METHOD_CODE (1..6), MULTIPLE_CODE (7) for multi-flag cells.
    """
    methods   = [m for m in METHOD_ORDER if m in mask_dict]
    shape     = next(iter(mask_dict.values())).shape
    combined  = np.zeros(shape, dtype=int)
    for name in methods:
        combined[mask_dict[name]] = METHOD_CODE[name]
    n_flags = sum(mask_dict[name].astype(int) for name in methods)
    combined[n_flags > 1] = MULTIPLE_CODE  # overwrite with 'multiple'
    return combined


def plot_rejection_heatmap(combined_matrix, ch_names, hypno_epochs, file_id, custom_stages=()):
    """
    Plot a channels x epochs heatmap coloured by rejection method.
    Top strip: hypnogram as a YASA-style step-line (W at top, N3 at bottom).
    Gray line for all stages; REM highlighted in red.
    Returns a matplotlib Figure.
    """
    # Index = method code: 0 none, 1 amplitude, 2 flat, 3 gradient, 4 1/f error, 5 1/f R2,
    # 6 event, 7 multiple (matches METHOD_CODE / MULTIPLE_CODE). 'event' colour unused when off.
    # 'event' = magenta #e84393: distinct from the green/blue/orange/red family and well legible
    # on the dark-purple 'none' background.
    COLORS = ['#1c0a3b', '#c0392b', '#2980b9', '#e67e22', '#f1c40f', '#27ae60', '#e84393', '#7b0000']
    LABELS = ['none', 'amplitude', 'flat', 'gradient', '1/f error', '1/f R2', 'event', 'multiple']
    cmap   = mcolors.ListedColormap(COLORS)
    bounds = np.arange(-0.5, 8.5, 1)
    norm   = mcolors.BoundaryNorm(bounds, cmap.N)

    # Hypnogram heights: W at top (4), N3 at bottom (0); custom stages stacked below N3.
    STAGE_H, _stage_colors, _ytick_pos, _ytick_labels = custom_stage_style(custom_stages)
    _floor = min(_ytick_pos) - 1
    hypno_y = np.array([STAGE_H.get(s, _floor) for s in hypno_epochs])

    n_channels   = len(ch_names)
    n_epochs     = combined_matrix.shape[0]
    pct_rejected = 100.0 * combined_matrix.any(axis=1).mean()

    fig_w = max(8, min(n_epochs / 6, 20))
    # Fixed-ish height: rows get thinner as the montage grows (capped so a 32-channel Curry
    # montage stays compact instead of a ~16 in tall figure). Unchanged for <=14 channels.
    fig_h = max(3, min(0.45 * n_channels + 2.0, 8.0))
    fig, axes = plt.subplots(
        2, 1, figsize=(fig_w, fig_h),
        gridspec_kw={'height_ratios': [1.2, n_channels], 'hspace': 0.04}
    )

    # --- Hypnogram strip ---
    ax_hyp = axes[0]
    x_step = np.arange(n_epochs + 1)
    y_step = np.append(hypno_y, hypno_y[-1])

    # Full step line in gray
    ax_hyp.step(x_step, y_step, where='post', color='#555555', linewidth=1.2)

    # REM in red; each custom stage highlighted in its own colour.
    for _s in (['R'] + list(custom_stages)):
        _col = '#c0392b' if _s == 'R' else _stage_colors.get(_s)
        for ei in range(n_epochs):
            if hypno_epochs[ei] == _s:
                ax_hyp.plot([ei, ei + 1], [STAGE_H[_s], STAGE_H[_s]],
                            color=_col, linewidth=2.5, solid_capstyle='butt')

    ax_hyp.set_yticks(_ytick_pos)
    ax_hyp.set_yticklabels(_ytick_labels, fontsize=7)
    ax_hyp.set_ylim(min(_ytick_pos) - 0.8, max(_ytick_pos) + 0.8)
    ax_hyp.set_xlim(0, n_epochs)
    ax_hyp.set_xticks([])
    for sp in ['top', 'right', 'bottom']:
        ax_hyp.spines[sp].set_visible(False)

    # --- Main heatmap ---
    ax = axes[1]
    im = ax.pcolormesh(
        np.arange(n_epochs + 1), np.arange(n_channels + 1),
        combined_matrix.T,  # (n_channels, n_epochs)
        cmap=cmap, norm=norm, rasterized=True
    )
    ax.set_yticks(np.arange(n_channels) + 0.5)
    ax.set_yticklabels(ch_names, fontsize=max(5, min(8, 200 / n_channels)))
    ax.set_xlabel('Epoch index', fontsize=9)
    ax.set_xlim(0, n_epochs)
    ax.set_ylim(0, n_channels)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

    cbar = fig.colorbar(im, ax=ax, orientation='vertical',
                        fraction=0.02, pad=0.01, ticks=np.arange(len(LABELS)))
    cbar.ax.set_yticklabels(LABELS, fontsize=7)

    fig.suptitle(f'{file_id} — Artefacts ({pct_rejected:.0f}% rejected)',
                 fontsize=11, y=1.01)
    fig.tight_layout()
    return fig


def build_rejection_summary(mask_dict, hypno_epochs, file_id, ch_names, custom_stages=()):
    """Per-stage, per-method rejection counts DataFrame (summary statistics)."""
    methods = [m for m in METHOD_ORDER if m in mask_dict]
    rows    = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages):
        stage_mask = (hypno_epochs == stage)
        n_total    = int(stage_mask.sum())
        any_flag   = np.zeros(len(hypno_epochs), dtype=bool)
        for name in methods:
            any_flag |= mask_dict[name].any(axis=1)
        n_rej_any = int((any_flag & stage_mask).sum())
        rows.append({
            'file_id': file_id, 'stage': stage, 'method': 'any',
            'n_total': n_total, 'n_rejected': n_rej_any,
            'pct_rejected': round(100 * n_rej_any / n_total, 1) if n_total > 0 else float('nan'),
        })
        for name in methods:
            per_epoch = mask_dict[name].any(axis=1)
            n_rej = int((per_epoch & stage_mask).sum())
            rows.append({
                'file_id': file_id, 'stage': stage, 'method': name,
                'n_total': n_total, 'n_rejected': n_rej,
                'pct_rejected': round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
            })
    return pd.DataFrame(rows)


def build_epoch_rejection_log(mask_dict, hypno_epochs, file_id):
    """Per-epoch rejection log: one row per epoch with epoch-level (any-channel) method flags."""
    methods  = [m for m in METHOD_ORDER if m in mask_dict]
    n_epochs = len(hypno_epochs)
    rows = []
    for ei in range(n_epochs):
        row = {
            'file_id':     file_id,
            'epoch_idx':   ei,
            'stage':       hypno_epochs[ei],
            'reject_flag': bool(any(mask_dict[m][ei].any() for m in methods)),
        }
        for m in methods:
            col = 'flag_' + m.replace('/', '').replace(' ', '')
            row[col] = bool(mask_dict[m][ei].any())
        rows.append(row)
    return pd.DataFrame(rows)


def build_global_summary_table(all_rejection_summaries, custom_stages=()):
    """
    Build a dataset-level pivot table from per-participant rejection summaries.
    One row per sleep stage; columns: n_total, n_rejected, pct_rejected (pooled),
    then n_rej_{method} and pct_rej_{method} for each rejection method.
    """
    df_all  = pd.concat(all_rejection_summaries, ignore_index=True)
    # Methods present in the loaded summaries (so 'event' columns appear only when used and
    # mixed runs — some with, some without event rejection — never raise).
    methods_list = [m for m in METHOD_ORDER if m in set(df_all['method'])]
    rows = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages):
        stage_data = df_all[df_all['stage'] == stage]
        any_data   = stage_data[stage_data['method'] == 'any']
        n_total    = int(any_data['n_total'].sum())
        n_rej      = int(any_data['n_rejected'].sum())
        n_parts    = int(len(any_data))
        row = {
            'stage':          stage,
            'n_participants': n_parts,
            'n_total':        n_total,
            'n_rejected':     n_rej,
            'pct_rejected':   round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
        }
        for m in methods_list:
            m_data  = stage_data[stage_data['method'] == m]
            n_m_rej = int(m_data['n_rejected'].sum())
            row[f'n_rej_{m}']   = n_m_rej
            row[f'pct_rej_{m}'] = round(100 * n_m_rej / n_total, 1) if n_total > 0 else float('nan')
        rows.append(row)
    return pd.DataFrame(rows)


def build_stage_method_html(rej_summary):
    """Per-stage rejection table for the HTML report: one row/stage, one column/method.

    Each method cell shows the % of that stage's epochs it flagged (in front) with the raw
    count in parentheses. 'event' appears only when event rejection ran (methods present in
    rej_summary). % is already relative to the stage total (n_total), matching
    global_rejection_by_stage.tsv.
    """
    if rej_summary is None or rej_summary.empty:
        return '<p><em>Per-stage summary unavailable.</em></p>'
    methods = [m for m in METHOD_ORDER if m in set(rej_summary['method'])]
    def _cell(pct, n):
        if pd.isna(pct):
            return '<td style="padding:3px 10px;text-align:right;">—</td>'
        return (f'<td style="padding:3px 10px;text-align:right;"><b>{pct:.1f}%</b>'
                f'<small style="color:#888;"> ({int(n)})</small></td>')
    head = ('<tr><th style="padding:3px 10px;text-align:left;">Stage</th>'
            '<th style="padding:3px 10px;text-align:right;">n epochs</th>'
            '<th style="padding:3px 10px;text-align:right;">Rejected (any)</th>'
            + ''.join(f'<th style="padding:3px 10px;text-align:right;">{METHOD_LABEL.get(m, m)}</th>'
                      for m in methods) + '</tr>')
    body = ''
    for stage in list(rej_summary['stage'].drop_duplicates()):
        rows_s  = rej_summary[rej_summary['stage'] == stage]
        any_row = rows_s[rows_s['method'] == 'any']
        if any_row.empty:
            continue
        n_total = int(any_row['n_total'].iloc[0])
        body += (f'<tr><td style="padding:3px 10px;">{stage}</td>'
                 f'<td style="padding:3px 10px;text-align:right;">{n_total}</td>'
                 + _cell(any_row['pct_rejected'].iloc[0], any_row['n_rejected'].iloc[0]))
        for m in methods:
            m_row = rows_s[rows_s['method'] == m]
            body += (_cell(m_row['pct_rejected'].iloc[0], m_row['n_rejected'].iloc[0])
                     if not m_row.empty else '<td style="padding:3px 10px;text-align:right;">—</td>')
        body += '</tr>'
    return f'<table style="border-collapse:collapse;font-size:.9em;">{head}{body}</table>'


def build_channel_stage_html(mask_dict, hypno_epochs, ch_names, custom_stages=()):
    """Per-channel × per-stage rejection table for the HTML report: channels as rows, stages
    as columns. Each cell = % of that stage's epochs where the channel was flagged by ANY
    method, with the raw count in parentheses; the last 'All' column pools all stages. Built
    from the per-(epoch, channel) masks (not persisted to TSV) — report-only."""
    methods = [m for m in METHOD_ORDER if m in mask_dict]
    if not methods:
        return '<p><em>Per-channel summary unavailable.</em></p>'
    hyp = np.asarray(hypno_epochs)
    any_ec = np.zeros(mask_dict[methods[0]].shape, dtype=bool)   # (n_epochs, n_channels) any-method
    for m in methods:
        any_ec |= mask_dict[m]
    stages = ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages)
    def _cell(n_rej, n_tot):
        if not n_tot:
            return '<td style="padding:3px 10px;text-align:right;">—</td>'
        return (f'<td style="padding:3px 10px;text-align:right;"><b>{100 * n_rej / n_tot:.1f}%</b>'
                f'<small style="color:#888;"> ({int(n_rej)})</small></td>')
    head = ('<tr><th style="padding:3px 10px;text-align:left;">Channel</th>'
            + ''.join(f'<th style="padding:3px 10px;text-align:right;">{s}</th>' for s in stages)
            + '<th style="padding:3px 10px;text-align:right;">All</th></tr>')
    body = ''
    for ci, ch in enumerate(ch_names):
        cells = ''
        for s in stages:
            stage_mask = (hyp == s)
            cells += _cell(int((any_ec[:, ci] & stage_mask).sum()), int(stage_mask.sum()))
        cells += _cell(int(any_ec[:, ci].sum()), len(hyp))
        body += f'<tr><td style="padding:3px 10px;">{ch}</td>{cells}</tr>'
    return f'<table style="border-collapse:collapse;font-size:.9em;">{head}{body}</table>'


def build_event_type_html(event_type_masks, hypno_epochs, custom_stages=()):
    """Per-event-type rejection table for the HTML report (event rejection only).

    One row per selected canonical event type + a bold '(any selected)' union row; columns:
    n epochs flagged, % of all epochs, then % (n) per sleep stage. Helps spot event types that
    reject too many epochs. Onset-only containment, same rule as compute_event_epoch_mask.
    """
    if not event_type_masks:
        return None
    stages   = ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages)
    hypno    = np.asarray(hypno_epochs)
    n_epochs = len(hypno)
    stage_totals = {s: int((hypno == s).sum()) for s in stages}
    def _pct(n, tot):
        return f'{100 * n / tot:.1f}%' if tot else '—'
    def _row(label, mask, bold=False):
        mask = np.asarray(mask, dtype=bool)
        n    = int(mask.sum())
        weight = 'font-weight:600;' if bold else ''
        cells = ''
        for s in stages:
            if stage_totals[s]:
                ns = int((mask & (hypno == s)).sum())
                cells += (f'<td style="padding:3px 10px;text-align:right;">{_pct(ns, stage_totals[s])}'
                          f'<small style="color:#888;"> ({ns})</small></td>')
            else:
                cells += '<td style="padding:3px 10px;text-align:right;">—</td>'
        return (f'<tr style="{weight}"><td style="padding:3px 10px;">{label}</td>'
                f'<td style="padding:3px 10px;text-align:right;">{n}</td>'
                f'<td style="padding:3px 10px;text-align:right;">{_pct(n, n_epochs)}</td>{cells}</tr>')
    any_mask = np.zeros(n_epochs, dtype=bool)
    for t in sorted(event_type_masks):
        any_mask |= np.asarray(event_type_masks[t], dtype=bool)
    head = ''.join(f'<th style="padding:3px 10px;text-align:right;">{s}</th>' for s in stages)
    body = ''.join(_row(t, event_type_masks[t]) for t in sorted(event_type_masks))
    body += _row('(any selected)', any_mask, bold=True)
    return ('<p style="font-size:.85em;color:#666;">Onset-only: each event flags the 30 s epoch '
            'containing its Start; annotated durations are ignored.</p>'
            '<table style="border-collapse:collapse;font-size:.9em;">'
            '<tr><th style="padding:3px 10px;text-align:left;">Event type</th>'
            '<th style="padding:3px 10px;text-align:right;">n epochs</th>'
            '<th style="padding:3px 10px;text-align:right;">%</th>'
            f'{head}</tr>{body}</table>')


## 1 — Paths

In [ ]:
fc_curry = FileChooser()
fc_curry.title = '<b>Select your data folder:</b>'
fc_curry.show_only_dirs = True

# Show only the aggregated quality_summary.tsv, not the per-file *_quality_metrics.tsv /
# *_quality_by_stage.tsv that quality_overview also writes into this folder.
fc_quality = FileChooser(filter_pattern='quality_summary.tsv')
fc_quality.title = '<b>quality_summary.tsv</b> (from quality_overview) — optional, recommended :'

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>remap_reref_persubject.json</b> (from select&amp;remap_channels_curry) :'

fc_events = FileChooser(filter_pattern='*.json')
fc_events.title = '<b>event_remap.json</b> (from remap_events) — optional, for event-based rejection :'

csv_suffix = widgets.Text(
    value='_ScoredEvents_Export.txt',
    description='Event export suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
csv_suffix_info = widgets.HTML(value='')

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Hypno suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')

custom_stages_box = widgets.Text(
    value='',
    description='Custom stages:',
    placeholder='non-AASM labels to keep, e.g. N4 (auto-filled from config)',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
custom_stages_info = widgets.HTML(value='')


def _detect_hypno_suffixes(chooser):
    """Auto-detect hypnogram .txt suffixes in the .cdt folder and populate the widget."""
    try:
        if not chooser.selected:
            hypno_suffix_info.value = ''
            csv_suffix_info.value = ''
            fc_quality.reset()
            fc_config.reset()
            fc_events.reset()
            fc_out.reset()
            return
        curry_folder = Path(chooser.selected)
        fc_quality.reset(path=str(curry_folder))
        fc_config.reset(path=str(curry_folder))
        _cp_dir = curry_folder / 'config_param'
        fc_events.reset(path=str(_cp_dir if _cp_dir.is_dir() else curry_folder))
        fc_out.reset(path=str(curry_folder))
        _cs = load_custom_stages(curry_folder)
        custom_stages_box.value = ', '.join(_cs)
        custom_stages_info.value = (
            f'<small style="color:#2e7d32;">Custom stage(s) loaded: {", ".join(_cs)}</small>' if _cs
            else '<small style="color:#888;">No custom stages registered for this dataset.</small>'
        )
        curry_files  = [f for f in sorted(curry_folder.rglob('*')) if f.suffix == '.cdt' and not f.name.startswith('._')]
        if not curry_files:
            hypno_suffix_info.value = (
                '<small style="color:#888;">No .cdt files found in selected folder (recursive scan)</small>'
            )
            return
        n_total = len(curry_files)
        # ---- Event export suffix auto-detection (Curry: *_ScoredEvents_Export.txt) ----
        # Scan .txt files whose name contains 'event' so hypnogram .txt files are not counted.
        all_evt = [f for f in curry_folder.rglob('*')
                   if f.suffix.lower() == '.txt' and 'event' in f.name.lower()]
        evt_counts = {}
        for cdt in curry_files:
            for c in all_evt:
                if os.path.normcase(c.name).startswith(os.path.normcase(cdt.stem)):
                    suf = c.name[len(cdt.stem):]
                    evt_counts[suf] = evt_counts.get(suf, 0) + 1
        if evt_counts:
            best_evt, best_evt_n = min(evt_counts.items(), key=lambda x: (-x[1], len(x[0])))
            csv_suffix.value = best_evt
            cparts = [f'<b>{s}</b>&nbsp;(×{c})' for s, c in sorted(evt_counts.items(), key=lambda x: -x[1])]
            ccolor = '#2e7d32' if best_evt_n == n_total else '#e67e00'
            csv_suffix_info.value = (
                f'<small style="color:{ccolor};">Event export detected: &nbsp;{"&nbsp;·&nbsp;".join(cparts)}'
                f'&nbsp;— {best_evt_n}/{n_total} files matching</small>'
            )
        else:
            csv_suffix_info.value = (
                '<small style="color:#888;">No event export detected next to the .cdt files — '
                'event-based rejection will be unavailable.</small>'
            )
        all_txt = [f for f in curry_folder.rglob('*') if f.suffix.lower() == '.txt']
        suffix_counts = {}
        for cdt in curry_files:
            for txt in all_txt:
                if os.path.normcase(txt.name).startswith(os.path.normcase(cdt.stem)):
                    suffix = txt.name[len(cdt.stem):]
                    suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
        if not suffix_counts:
            hypno_suffix_info.value = (
                '<small style="color:#e67e00;">No hypnogram .txt files detected — '
                'please verify that the files are present or adjust the suffix manually.</small>'
            )
            return
        # Among suffixes appearing for >=50% of max count, prefer the longest
        # (more specific = remapped/processed version). Tiebreaker: highest count.
        # Auto-selection skips event exports: Curry writes its scored events to
        # *_ScoredEvents_Export.txt, a .txt sitting next to the hypnograms whose suffix is
        # LONGER than _Hypnogram_remapped.txt, so 'prefer the longest' would select it and try
        # to read event lines as sleep stages. Excluding 'event' rather than requiring 'hypno'
        # leaves hypnogram naming unconstrained. Filtered out of the SELECTION only — every
        # .txt suffix stays listed below, so a wrong detection stays visible and correctable.
        sel_counts = {s: c for s, c in suffix_counts.items() if 'event' not in s.lower()}
        if not sel_counts:
            sel_counts = suffix_counts  # only event exports found: fall back rather than fail
        max_count = max(sel_counts.values())
        candidates = {s: c for s, c in sel_counts.items() if c >= max_count * 0.5}
        best_suffix, best_count = max(candidates.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c}){"&nbsp;← selected" if s == best_suffix else ""}'
                 for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Detected: &nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} files matching</small>'
        )
    except Exception as e:
        hypno_suffix_info.value = (
            f'<small style="color:#c0392b;">Error detecting hypnograms: {e}</small>'
        )
    _update_existing_reports_info()


def _update_existing_reports_info(chooser=None):
    if not fc_curry.selected or not fc_out.selected:
        existing_reports_info.value = ''
        return
    try:
        curry_folder = Path(fc_curry.selected)
        out_root = Path(fc_out.selected)
        curry_files = [f for f in sorted(curry_folder.rglob('*')) if f.suffix == '.cdt' and not f.name.startswith('._')]
        n_total = len(curry_files)
        if n_total == 0:
            existing_reports_info.value = '<small style="color:#888;">No .cdt files found in selected folder.</small>'
            return
        reports_dir = out_root / 'reports_preprocessing'
        n_existing = sum(
            1 for f in curry_files
            if (reports_dir / f'{f.stem}_preprocessing_report.html').exists()
        )
        existing_reports_info.value = (
            f'<small style="color:#555;">'
            f'{n_existing} / {n_total} participant(s) already have an existing preprocessing report.</small>'
        )
    except Exception as e:
        existing_reports_info.value = f'<small style="color:#c0392b;">Error checking existing reports: {e}</small>'


fc_curry.register_callback(_detect_hypno_suffixes)

existing_reports_info = widgets.HTML(value='')

skip_existing = widgets.Checkbox(
    value=True,
    description='Skip participants with an existing report',
    style={'description_width': 'initial'}
)

fc_out = FileChooser()
fc_out.title = '<b>Output folder</b> (will receive <em>reports_preprocessing/</em> and <em>derivatives/</em>) :'
fc_out.show_only_dirs = True
fc_out.register_callback(_update_existing_reports_info)


def _populate_event_types(chooser=None):
    """Rebuild one checkbox per canonical event type from event_remap.json's values."""
    if 'event_types_box' not in globals():
        return
    event_type_checkboxes.clear()
    try:
        if fc_events.selected:
            types = sorted({v for v in load_event_remap(fc_events.selected).values() if v})
        else:
            types = []
    except Exception:
        types = []
    if not types:
        event_types_box.children = [event_types_hint]
        return
    for t in types:
        event_type_checkboxes[t] = widgets.Checkbox(
            value=False, description=t, indent=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='auto', margin='2px 14px 2px 0'))
    event_types_box.children = list(event_type_checkboxes.values())


fc_events.register_callback(_populate_event_types)

display(
    fc_curry,
    hypno_suffix,
    hypno_suffix_info,
    custom_stages_box,
    custom_stages_info,
    fc_config,
    fc_quality,
    fc_events,
    csv_suffix,
    csv_suffix_info,
    fc_out,
    existing_reports_info,
    skip_existing,
)

## 2 — Preprocessing and rejection parameters

In [ ]:
# ---- Resampling ----
cb_resample = widgets.Checkbox(
    value=False, description='Activate resampling',
    style={'description_width': 'initial'}
)
txt_target_freq = widgets.BoundedIntText(
    value=256, min=1, max=10000, step=1,
    description='Target frequency (Hz) :',
    style={'description_width': '180px'},
    # Initial visibility follows the checkbox default: the observer below only fires on a
    # CHANGE, so a box hard-coded to 'none' would stay hidden under a ticked checkbox.
    layout=widgets.Layout(width='320px', display='' if cb_resample.value else 'none')
)

def _toggle_resample(change):
    txt_target_freq.layout.display = '' if change['new'] else 'none'

cb_resample.observe(_toggle_resample, names='value')

# ---- Filter ----
cb_filter = widgets.Checkbox(
    value=True, description='Activate bandpass filter',
    style={'description_width': 'initial'}
)
txt_l_freq = widgets.BoundedFloatText(
    value=0.1, min=0.0, max=100.0, step=0.05,
    description='l_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)
txt_h_freq = widgets.BoundedFloatText(
    value=50.0, min=1.0, max=500.0, step=1.0,
    description='h_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)

# ---- Notch filter ----
cb_notch = widgets.Checkbox(
    value=False, description='Activate notch filter',
    style={'description_width': 'initial'}
)
txt_notch_freq = widgets.BoundedFloatText(
    value=50.0, min=1.0, max=500.0, step=1.0,
    description='Notch freq (Hz) :',
    style={'description_width': '120px'},
    # Initial visibility follows the checkbox default (same reason as the resampling box above).
    layout=widgets.Layout(width='260px', display='' if cb_notch.value else 'none')
)

def _toggle_notch(change):
    txt_notch_freq.layout.display = '' if change['new'] else 'none'

cb_notch.observe(_toggle_notch, names='value')

# ---- Epoch rejection thresholds ----
_ws = {'description_width': '200px'}
_wl = widgets.Layout(width='360px')

txt_thresh_W  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold W (µV):', style=_ws, layout=_wl)
txt_thresh_N1 = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N1 (µV):', style=_ws, layout=_wl)
txt_thresh_N2 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N2 (µV):', style=_ws, layout=_wl)
txt_thresh_N3 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N3 (µV):', style=_ws, layout=_wl)
txt_thresh_R  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold REM (µV):', style=_ws, layout=_wl)

# Per-custom-stage amplitude thresholds (rebuilt when the 'Custom stages' field changes).
custom_thresh_widgets = {}
custom_thresh_box = widgets.VBox([])


def rebuild_custom_thresholds(*_):
    cs = parse_custom_field(custom_stages_box.value)
    rows, new_widgets = [], {}
    for s in cs:
        w = custom_thresh_widgets.get(s) or widgets.BoundedFloatText(
            value=250.0, min=1.0, max=5000.0, step=10.0,
            description=f'Amplitude threshold {s} (µV):', style=_ws, layout=_wl)
        new_widgets[s] = w
        rows.append(w)
    custom_thresh_widgets.clear()
    custom_thresh_widgets.update(new_widgets)
    custom_thresh_box.children = tuple(rows)


custom_stages_box.observe(rebuild_custom_thresholds, names='value')
rebuild_custom_thresholds()
txt_flat      = widgets.BoundedFloatText(value=1.0, min=0.01, max=100.0, step=0.1,
    description='Flat signal < (µV):', style=_ws, layout=_wl)
txt_grad      = widgets.BoundedFloatText(value=100.0, min=1.0, max=5000.0, step=10.0,
    description='Max gradient (µV/sample):', style=_ws, layout=_wl)
txt_1f_error  = widgets.BoundedFloatText(value=0.15, min=0.0, max=1.0, step=0.01,
    description='1/f error threshold >:', style=_ws, layout=_wl)
txt_1f_r2     = widgets.BoundedFloatText(value=0.85, min=0.0, max=1.0, step=0.01,
    description='1/f R² threshold <:', style=_ws, layout=_wl)
# Frequency range over which the aperiodic (1/f) model is fitted. Default 2–45 Hz: the low
# bound (2 Hz) limits slow-wave influence, the high bound (45 Hz) stays below the 50 Hz line and
# below Nyquist for typical resampled rates. The Welch PSD is computed up to this max.
txt_1f_fmin   = widgets.BoundedFloatText(value=2.0, min=0.1, max=100.0, step=0.5,
    description='1/f fit min (Hz):', style=_ws, layout=_wl)
txt_1f_fmax   = widgets.BoundedFloatText(value=45.0, min=1.0, max=200.0, step=0.5,
    description='1/f fit max (Hz):', style=_ws, layout=_wl)


# ---- Event-based rejection ----
cb_event_reject = widgets.Checkbox(
    value=False, description='Reject epochs containing scored events',
    style={'description_width': 'initial'}
)
event_explain = widgets.HTML(
    value='<small style="color:#555;">An epoch is rejected when the <b>onset</b> of a '
          'checked event falls inside it. The annotated event <b>duration is ignored</b> '
          '(often only the onset is annotated, not the precise duration), so each event '
          'flags the single 30&nbsp;s epoch that contains its start time.</small>'
)
event_types_hint = widgets.HTML(
    value='<small style="color:#888;">Select an <b>event_remap.json</b> in Section 1 to '
          'populate the event types.</small>'
)
event_type_checkboxes = {}   # canonical label -> Checkbox (rebuilt from event_remap.json)

def selected_event_types():
    """Canonical event labels currently checked (empty list when none / unpopulated)."""
    return [lbl for lbl, cb in event_type_checkboxes.items() if cb.value]

# All event types are shown at once as a wrapping row of checkboxes (no scroll box).
event_types_box = widgets.Box([event_types_hint],
    layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
btn_count_events = widgets.Button(description='Count affected epochs',
    layout=widgets.Layout(width='220px'))
out_count_events = widgets.Output()
event_box = widgets.VBox([
    event_explain, event_types_box,
    btn_count_events, out_count_events,
])

def _toggle_event_reject(change):
    event_box.layout.display = '' if change['new'] else 'none'

_toggle_event_reject({'new': cb_event_reject.value})
cb_event_reject.observe(_toggle_event_reject, names='value')


def _count_affected_epochs(btn):
    # Count, over the participants CHECKED in Section 3, how many 30 s epochs each selected
    # event type would flag (no signal is read — only the hypnogram length/stages + events).
    out_count_events.clear_output()
    with out_count_events:
        if 'participant_ui' not in globals() or not participant_ui:
            print('Load participants first (Section 3), then check the ones to count.')
            return
        checked = [fid for fid, ui in participant_ui.items() if ui['cb'].value]
        if not checked:
            print('No participant checked in Section 3.')
            return
        if not fc_events.selected:
            print('Select an event_remap.json in Section 1 first.')
            return
        types = selected_event_types()
        if not types:
            print('Select at least one event type.')
            return
        if not fc_curry.selected:
            print('Select the .cdt data folder in Section 1 first.')
            return
        try:
            event_remap = load_event_remap(fc_events.selected)
        except Exception as e:
            print(f'Could not read event_remap.json: {e}')
            return
        curry_folder = Path(fc_curry.selected)
        hypno_lookup = {os.path.normcase(p.name): p for p in curry_folder.rglob('*') if p.suffix.lower() == '.txt'}
        stages = ['W', 'N1', 'N2', 'N3', 'R'] + parse_custom_field(custom_stages_box.value)
        rowtypes = types + ['(any selected)']
        counts = {t: {'n': 0, 'stage': {s: 0 for s in stages}} for t in rowtypes}
        total_epochs = 0
        stage_totals = {s: 0 for s in stages}
        n_files_ok = 0
        n_no_events = 0
        for fid in checked:
            cdt_cand = [f for f in curry_folder.rglob('*')
                        if os.path.normcase(f.stem) == os.path.normcase(fid) and f.suffix == '.cdt']
            if not cdt_cand:
                continue
            hp = hypno_lookup.get(os.path.normcase(f'{fid}{hypno_suffix.value}'))
            if hp is None or not hp.exists():
                continue
            try:
                hyp = np.loadtxt(str(hp), dtype=str).astype('<U10')
            except Exception:
                continue
            n_ep = len(hyp)
            events_df, _src = load_events(cdt_cand[0], csv_suffix.value)
            if events_df is None:
                n_no_events += 1
                continue
            n_files_ok += 1
            total_epochs += n_ep
            for s in stages:
                stage_totals[s] += int((hyp == s).sum())
            any_mask = np.zeros(n_ep, dtype=bool)
            for t in types:
                m = compute_event_epoch_mask(events_df, event_remap, [t], n_ep, 30.0)
                counts[t]['n'] += int(m.sum())
                for s in stages:
                    counts[t]['stage'][s] += int((m & (hyp == s)).sum())
                any_mask |= m
            counts['(any selected)']['n'] += int(any_mask.sum())
            for s in stages:
                counts['(any selected)']['stage'][s] += int((any_mask & (hyp == s)).sum())
        if n_files_ok == 0:
            print(f'No event companions found for the checked participants '
                  f'(checked {len(checked)}, {n_no_events} without events).')
            return
        def _pct(n, tot):
            return f'{100 * n / tot:.1f}%' if tot else '—'
        head = ''.join(f'<th style="padding:3px 10px;text-align:right;">{s}</th>' for s in stages)
        body = ''
        for t in rowtypes:
            c = counts[t]
            per_stage = ''.join(
                (f'<td style="padding:3px 10px;text-align:right;">{_pct(c["stage"][s], stage_totals[s])}'
                 f'<small style="color:#888;"> ({c["stage"][s]})</small></td>')
                if stage_totals[s] else '<td style="padding:3px 10px;text-align:right;">—</td>'
                for s in stages
            )
            weight = 'font-weight:600;' if t == '(any selected)' else ''
            body += (f'<tr style="{weight}"><td style="padding:3px 10px;">{t}</td>'
                     f'<td style="padding:3px 10px;text-align:right;">{c["n"]}</td>'
                     f'<td style="padding:3px 10px;text-align:right;">{_pct(c["n"], total_epochs)}</td>'
                     f'{per_stage}</tr>')
        html = (
            f'<p>{n_files_ok} participant(s) with events · {total_epochs} epochs total'
            + (f' · {n_no_events} without event companion' if n_no_events else '') + '</p>'
            '<table style="border-collapse:collapse;font-size:.9em;">'
            '<tr><th style="padding:3px 10px;text-align:left;">Event type</th>'
            '<th style="padding:3px 10px;text-align:right;">n epochs</th>'
            '<th style="padding:3px 10px;text-align:right;">%</th>'
            f'{head}</tr>{body}</table>'
        )
        display(HTML(html))

btn_count_events.on_click(_count_affected_epochs)


display(
    HTML('<h3 style="margin:8px 0 4px;">Preprocessing</h3>'),
    HTML('<b>Resampling</b>'),
    cb_resample, txt_target_freq,
    HTML('<br><b>Bandpass filter</b>'),
    cb_filter, widgets.HBox([txt_l_freq, txt_h_freq]),
    HTML('<br><b>Notch filter</b>'),
    cb_notch, txt_notch_freq,
    HTML('<hr style="margin:14px 0 4px;"><h3 style="margin:8px 0 4px;">Epoch rejection</h3>'),
    HTML('<b>Peak-to-peak amplitude per stage</b>'),
    widgets.HBox([txt_thresh_W, txt_thresh_N1]),
    widgets.HBox([txt_thresh_N2, txt_thresh_N3]),
    txt_thresh_R,
    custom_thresh_box,
    HTML('<br><b>Flat signal &amp; gradient</b>'),
    widgets.HBox([txt_flat, txt_grad]),
    HTML('<br><b>1/f fit quality (specparam)</b>'),
    widgets.HBox([txt_1f_error, txt_1f_r2]),
    widgets.HBox([txt_1f_fmin, txt_1f_fmax]),
    HTML('<br><b>Reject epochs containing events</b>'),
    cb_event_reject, event_box,
)

## 3 — Select participants & run preprocessing

In [ ]:
participant_ui         = {}   # file_id -> {'cb': Checkbox, 'channels': {ch: Checkbox}}
participant_ui_defaults = {}   # file_id -> {'cb': bool, 'channels': {ch: bool}}
_reset_in_progress      = [False]
vbox_participants       = widgets.VBox([])

btn_load  = widgets.Button(description='Load participants',
                           button_style='info',
                           layout=widgets.Layout(width='240px', height='36px'))
load_info = widgets.HTML(value='')


def on_load_participants(btn):
    btn.disabled = True
    # Ground truth = the EDF folder + the config JSON (both required). quality_summary.tsv is
    # an OPTIONAL QC-enrichment layer (pre-fills the per-channel exclude checkboxes); strongly
    # encouraged, but the tool can run from EDF + config alone (channels taken from the config
    # remap). This way a participant present on disk + in the config is NEVER hidden just
    # because it is missing from quality_summary.
    if not fc_curry.selected or not fc_config.selected:
        load_info.value = ('<span style="color:#c0392b;">'
                           'Please select the EDF folder and the JSON config first.</span>')
        btn.disabled = False
        return
    try:
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            cfg = json.load(_f)
    except Exception as e:
        load_info.value = f'<span style="color:#c0392b;">Error reading config JSON: {e}</span>'
        btn.disabled = False
        return

    # quality_summary is optional. If absent/unreadable, proceed without QC pre-selection but warn.
    # dtype={'file_id': str} keeps purely numeric IDs (e.g. 52, 117) as strings so they match
    # both JSON config keys (always str) and Path.stem (always str).
    df_q = None
    quality_warn = ''
    if fc_quality.selected:
        try:
            df_q = pd.read_csv(fc_quality.selected, sep='\t', dtype={'file_id': str})
        except Exception as e:
            quality_warn = (f'Could not read quality_summary.tsv ({e}) — proceeding without QC '
                            f'pre-selection.')
    else:
        quality_warn = ('No quality_summary.tsv selected — proceeding without QC pre-selection. '
                        'Running 5_quality_overview first is strongly recommended.')

    if df_q is not None and 'exclude' in df_q.columns:
        excl_all = df_q.groupby('file_id')['exclude'].all()
    else:
        excl_all = pd.Series(dtype=bool)
    q_ids = sorted(df_q['file_id'].astype(str).unique()) if df_q is not None else []

    # --- Discover the real recordings on disk (ground truth) ---
    curry_folder = Path(fc_curry.selected)
    rec_paths = [f for f in sorted(curry_folder.rglob('*'))
                 if f.suffix == '.cdt' and not f.name.startswith('._')]
    all_ids = [f.stem for f in rec_paths]

    # Normalized id sets for cross-source matching (normcase at the comparison only).
    cfg_idset = {os.path.normcase(k) for k in cfg}
    q_idset   = {os.path.normcase(s) for s in q_ids}
    edf_idset = {os.path.normcase(s) for s in all_ids}

    # --- Collect discrepancy warnings between the three sources ---
    disc_warnings = []
    if quality_warn:
        disc_warnings.append(quality_warn)
    if df_q is not None and len(all_ids) != len(q_ids):
        disc_warnings.append(f'{len(all_ids)} EDF file(s) detected vs {len(q_ids)} in quality_summary.tsv.')
    edf_not_in_q   = [s for s in all_ids if df_q is not None and os.path.normcase(s) not in q_idset]
    edf_not_in_cfg = [s for s in all_ids if os.path.normcase(s) not in cfg_idset]
    q_not_in_edf   = [s for s in q_ids if os.path.normcase(s) not in edf_idset]
    if edf_not_in_q:
        disc_warnings.append(f'{len(edf_not_in_q)} EDF(s) not in quality_summary — channels taken '
                             f'from the config JSON, no QC pre-selection: {", ".join(edf_not_in_q)}.')
    if edf_not_in_cfg:
        disc_warnings.append(f'{len(edf_not_in_cfg)} EDF(s) not in the config JSON — cannot be processed '
                             f'(run 2_select&remap_channels first): {", ".join(edf_not_in_cfg)}.')
    if q_not_in_edf:
        disc_warnings.append(f'{len(q_not_in_edf)} quality_summary entry(ies) with no matching EDF on '
                             f'disk (stale): {", ".join(q_not_in_edf)}.')

    # Filter out participants already FULLY processed when skip_existing is checked.
    # Fully processed = BOTH the HTML report AND the per-file epoch_rejection.tsv on disk.
    # If only one is present (run interrupted between the two, or a manual delete), the
    # participant is kept for reprocessing and a mismatch warning is collected below.
    n_hidden = 0
    skip_mismatches = []   # list of '{fid} ({reason})' strings
    if skip_existing.value and fc_out.selected:
        reports_dir_check = Path(fc_out.selected) / 'reports_preprocessing'
        ids_to_load = []
        for fid in all_ids:
            has_report = (reports_dir_check / f'{fid}_preprocessing_report.html').exists()
            has_data   = (reports_dir_check / f'{fid}_epoch_rejection.tsv').exists()
            if has_report and has_data:
                n_hidden += 1
            else:
                if has_report and not has_data:
                    skip_mismatches.append(f'{fid} (report but no epoch data)')
                elif has_data and not has_report:
                    skip_mismatches.append(f'{fid} (epoch data but no report)')
                ids_to_load.append(fid)
    else:
        ids_to_load = list(all_ids)

    participant_blocks = []
    global participant_ui, participant_ui_defaults
    participant_ui = {}
    participant_ui_defaults = {}

    # [J] Chaque participant est wrappé indépendamment pour ne pas bloquer les autres
    load_errors = []
    for fid in ids_to_load:
        try:
            # Resolve the config entry case-insensitively (normcase at the comparison only).
            cfg_key = next((k for k in cfg if os.path.normcase(k) == os.path.normcase(fid)), None)
            not_in_cfg     = cfg_key is None
            fully_excluded = bool(excl_all.get(fid, False))

            # Channel list + exclude flags: from quality_summary if this participant is in it,
            # otherwise the remapped channel names from the config montage (remap VALUES, not
            # keys — the UI and run loop work on the renamed channels), otherwise empty.
            in_quality = df_q is not None and os.path.normcase(fid) in q_idset
            src_note = ''
            if in_quality:
                fid_rows = df_q[df_q['file_id'].astype(str).map(os.path.normcase) == os.path.normcase(fid)]
                if 'channel' in df_q.columns and 'exclude' in df_q.columns:
                    chs_flags = {row['channel']: bool(row['exclude']) for _, row in fid_rows.iterrows()}
                elif 'channel' in df_q.columns:
                    chs_flags = {row['channel']: False for _, row in fid_rows.iterrows()}
                else:
                    chs_flags = {ch: False for ch in (cfg[cfg_key]['remap'].values() if cfg_key else [])}
            elif cfg_key is not None:
                chs_flags = {ch: False for ch in cfg[cfg_key].get('remap', {}).values()}
                src_note = ' (from config — no QC)'
            else:
                chs_flags = {}

            n_ch   = len(chs_flags)
            n_excl = sum(1 for excl in chs_flags.values() if excl)
            suffix_label = ''
            if fully_excluded:
                suffix_label = ' (all channels excluded)'
            elif not_in_cfg:
                suffix_label = ' (not found in JSON config)'
            elif src_note:
                suffix_label = src_note

            p_cb = widgets.Checkbox(
                value=(not fully_excluded and not not_in_cfg),
                description=f'{fid}{suffix_label}  [{n_ch} channels, {n_excl} excluded]',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='440px')
            )

            ch_cbs = {}
            for ch, excl in chs_flags.items():
                ch_cb = widgets.Checkbox(
                    value=not excl,
                    description=ch,
                    indent=False,
                    style={'description_width': 'initial'},
                    layout=widgets.Layout(width='120px')
                )
                ch_cbs[ch] = ch_cb

            ch_row = widgets.HBox(
                list(ch_cbs.values()),
                layout=widgets.Layout(flex_wrap='wrap', margin='0 0 10px 28px')
            )
            participant_ui[fid] = {'cb': p_cb, 'channels': ch_cbs}
            participant_ui_defaults[fid] = {
                'cb': p_cb.value,
                'channels': {ch: cb.value for ch, cb in ch_cbs.items()}
            }
            p_cb.observe(make_p_cb_observer(fid), names='value')
            participant_blocks.append(widgets.VBox([p_cb, ch_row]))
        except Exception as e:
            load_errors.append(str(fid))
            participant_blocks.append(widgets.HTML(
                value=f'<span style="color:#c0392b;">{fid} — loading error: {e}</span>'
            ))

    vbox_participants.children = participant_blocks
    n_ok = len(ids_to_load) - len(load_errors)
    msg = f'<span style="color:#2e7d32;">{n_ok} participant(s) loaded.</span>'
    if disc_warnings:
        msg += ('<br><span style="color:#e67e22;">⚠ '
                + '<br>⚠ '.join(disc_warnings) + '</span>')
    if n_hidden:
        msg += (f'&nbsp;&nbsp;<span style="color:#888;">'
                f'{n_hidden} hidden (report + data present).</span>')
    if skip_mismatches:
        msg += (f'&nbsp;&nbsp;<span style="color:#e67e22;">⚠ {len(skip_mismatches)} '
                f'inconsistent output(s) — will be reprocessed: {", ".join(skip_mismatches)}.</span>')
    if load_errors:
        msg += (f'&nbsp;&nbsp;<span style="color:#c0392b;">{len(load_errors)} error(s): '
                f'{", ".join(load_errors)}</span>')
    load_info.value = msg
    btn.disabled = False


btn_load.on_click(on_load_participants)


def make_p_cb_observer(fid):
    def _on_participant_toggle(change):
        if _reset_in_progress[0]:
            return
        for ch_cb in participant_ui[fid]['channels'].values():
            ch_cb.value = change['new']
    return _on_participant_toggle


btn_select_all   = widgets.Button(description='Select all',        layout=widgets.Layout(width='120px'))
btn_deselect_all = widgets.Button(description='Deselect all',      layout=widgets.Layout(width='130px'))
btn_reset_sel    = widgets.Button(description='Reset to defaults',  layout=widgets.Layout(width='160px'))
ctrl_row         = widgets.HBox([btn_select_all, btn_deselect_all, btn_reset_sel],
                                layout=widgets.Layout(margin='4px 0 8px 0'))


def _select_all(btn):
    for fid, ui in participant_ui.items():
        ui['cb'].value = True
        for ch_cb in ui['channels'].values():
            ch_cb.value = True


def _deselect_all(btn):
    for fid, ui in participant_ui.items():
        ui['cb'].value = False
        for ch_cb in ui['channels'].values():
            ch_cb.value = False


def _reset_to_defaults(btn):
    _reset_in_progress[0] = True
    for fid, ui in participant_ui.items():
        if fid in participant_ui_defaults:
            ui['cb'].value = participant_ui_defaults[fid]['cb']
            for ch, ch_cb in ui['channels'].items():
                if ch in participant_ui_defaults[fid]['channels']:
                    ch_cb.value = participant_ui_defaults[fid]['channels'][ch]
    _reset_in_progress[0] = False


btn_select_all.on_click(_select_all)
btn_deselect_all.on_click(_deselect_all)
btn_reset_sel.on_click(_reset_to_defaults)


# ---- Run button ----
btn_run      = widgets.Button(description='▶  Run preprocessing',
                              button_style='primary',
                              layout=widgets.Layout(width='260px', height='40px'))
progress     = widgets.IntProgress(min=0, max=1, value=0, bar_style='info',
                                   layout=widgets.Layout(width='500px'))
progress_lbl = widgets.Label(value='')
# --- Pipeline progress (2nd bar): spans ONE participant's whole run, so a long file is not
# mistaken for a crash. Its scale is in arbitrary 'time-cost' units (below) so the bar fill AND
# the 4-segment legend under it are sized ~proportionally to each phase. Load is one-time; the
# optional resample/notch/filter only count when enabled (recomputed per participant). Tune
# these if the segment widths feel off for your data.
COST_LOAD     = 8    # EDF read — one-time per file
COST_RESAMPLE = 4    # resampling — one-time, only when enabled
COST_NOTCH    = 3    # notch filter — one-time, only when enabled
COST_FILTER   = 3    # band-pass filter — one-time, only when enabled
COST_EPOCH    = 4    # epoching
COST_REJECT   = 10   # PSD + rejection masks (1/f fit) — the slowest step
COST_SAVE     = 5    # .fif + params + context companion
COST_REPORT   = 6    # heatmap + HTML report


def build_step_legend(c_load, c_epoch, c_reject, c_report):
    """4-segment strip drawn under the pipeline bar; segment widths ~ each phase's share of the
    run time (Load / Epoch / Reject / Report). Rebuilt per participant (widths depend on which
    optional steps run) and aligned to the 500px bar so the fill matches the segments."""
    total = max(1, c_load + c_epoch + c_reject + c_report)
    def seg(cost, label, bg, tip, last=False):
        border = '' if last else 'border-right:1px solid #bbb;'
        return (f'<div title="{tip}" style="width:{100 * cost / total:.1f}%;{border}'
                f'text-align:center;background:{bg};overflow:hidden;white-space:nowrap;">{label}</div>')
    return (
        '<div style="display:flex;width:500px;height:14px;font-size:9px;line-height:14px;'
        'color:#444;border:1px solid #bbb;border-top:none;border-radius:0 0 3px 3px;overflow:hidden;">'
        + seg(c_load, 'Load', '#e7ecff', 'Load / resample / filter')
        + seg(c_epoch, 'Epoch', '#eef7e7', 'Epoching')
        + seg(c_reject, 'Reject', '#e7f7e7', 'PSD + rejection (1/f fit)')
        + seg(c_report, 'Report', '#ffeaea', 'Save + report', last=True)
        + '</div>'
    )


progress_step = widgets.IntProgress(min=0, max=1, value=0, bar_style='',
                                    layout=widgets.Layout(width='500px', margin='0'))
step_legend   = widgets.HTML(value='', layout=widgets.Layout(margin='0'))
# Detail label to the RIGHT of the pipeline bar (mirrors tool 5): the 1st bar carries only the
# participant + i/N, this one names the current step (loading, resampling, epoching…).
progress_step_label = widgets.Label(value='')
out_run      = widgets.Output()


def run_preprocessing(btn):
    btn.disabled = True
    out_run.clear_output()

    # quality_summary.tsv is optional (Section 3 falls back to the config montage), so it is
    # not part of the required-paths guard — only the EDF folder, config JSON and output are.
    if not all([fc_curry.selected, fc_config.selected, fc_out.selected]):
        with out_run:
            print('ERROR: Please select the required paths (.cdt folder, JSON config, output). '
                  'quality_summary.tsv is optional but recommended.')
        btn.disabled = False
        return

    curry_folder = Path(fc_curry.selected)
    out_root   = Path(fc_out.selected)

    try:
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            config_dict = json.load(_f)
    except Exception as e:
        with out_run:
            print(f'ERROR loading config: {e}')
        btn.disabled = False
        return

    reports_dir = out_root / 'reports_preprocessing'
    try:
        reports_dir.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        with out_run:
            print(f'ERROR creating output folder: {e}')
        btn.disabled = False
        return

    do_resample      = cb_resample.value
    target_freq      = int(txt_target_freq.value) if do_resample else None
    do_filter        = cb_filter.value
    l_freq_val       = float(txt_l_freq.value)
    h_freq_val       = float(txt_h_freq.value)
    do_notch         = cb_notch.value
    notch_freq_val   = float(txt_notch_freq.value)
    ptp_thresh = {
        'W':  float(txt_thresh_W.value),
        'N1': float(txt_thresh_N1.value),
        'N2': float(txt_thresh_N2.value),
        'N3': float(txt_thresh_N3.value),
        'R':  float(txt_thresh_R.value),
    }
    custom_stages = parse_custom_field(custom_stages_box.value)
    for _cs, _cw in custom_thresh_widgets.items():
        ptp_thresh[_cs] = float(_cw.value)
    flat_thresh_val  = float(txt_flat.value)
    grad_thresh_val  = float(txt_grad.value)
    thresh_error_val = float(txt_1f_error.value)
    thresh_r2_val    = float(txt_1f_r2.value)
    fit_fmin_val     = float(txt_1f_fmin.value)
    fit_fmax_val     = float(txt_1f_fmax.value)
    if fit_fmax_val <= fit_fmin_val:
        with out_run:
            print(f'⚠ 1/f fit range invalid ({fit_fmin_val}–{fit_fmax_val} Hz); falling back to 2–45 Hz.')
        fit_fmin_val, fit_fmax_val = 2.0, 45.0
    do_event         = cb_event_reject.value
    event_types      = set(selected_event_types())
    event_remap_dict = {}
    if do_event:
        if not event_types:
            with out_run:
                print('⚠ Event-based rejection is on but no event type selected — disabled for this run.')
            do_event = False
        elif not fc_events.selected:
            with out_run:
                print('⚠ Event-based rejection is on but no event_remap.json selected — disabled for this run.')
            do_event = False
        else:
            try:
                event_remap_dict = load_event_remap(fc_events.selected)
            except Exception as e:
                with out_run:
                    print(f'⚠ Could not read event_remap.json ({e}) — event-based rejection disabled for this run.')
                do_event = False

    selected_ids = [fid for fid, ui in participant_ui.items() if ui['cb'].value]
    if not selected_ids:
        with out_run:
            print('No participants selected.')
        btn.disabled = False
        return

    progress.max   = len(selected_ids)
    progress.value = 0
    all_rejection_summaries = []
    all_epoch_logs          = []
    failed                  = []
    attempted_ids           = set()
    t_start                 = time.time()
    hypno_lookup = {os.path.normcase(p.name): p for p in curry_folder.rglob('*') if p.suffix.lower() == '.txt'}

    for idx, file_id in enumerate(selected_ids):
        # 1st bar = participant only (mirrors tool 5); the 2nd-bar label carries the current
        # phase (loading, resampling, epoching…) so the two bars are not redundant.
        progress_lbl.value = f'Participant {file_id}  ({idx + 1}/{len(selected_ids)})'
        def set_phase(msg):
            progress_step_label.value = f'{file_id} · {msg}'
        set_phase('starting…')
        # Pipeline bar (2nd bar): size it to this participant's active steps and (re)build the
        # 4-segment legend — the optional resample/notch/filter widths depend on this run.
        c_load_grp = (COST_LOAD + (COST_RESAMPLE if do_resample else 0)
                      + (COST_NOTCH if do_notch else 0) + (COST_FILTER if do_filter else 0))
        progress_step.max   = c_load_grp + COST_EPOCH + COST_REJECT + COST_SAVE + COST_REPORT
        progress_step.value = 0
        step_legend.value   = build_step_legend(c_load_grp, COST_EPOCH, COST_REJECT, COST_SAVE + COST_REPORT)

        # Skip only when BOTH the report and the per-file epoch data exist, so a run
        # interrupted between the two never leaves a participant permanently skipped with
        # missing data. A mismatch (only one present) is warned about and reprocessed.
        _has_report = (reports_dir / f'{file_id}_preprocessing_report.html').exists()
        _has_data   = (reports_dir / f'{file_id}_epoch_rejection.tsv').exists()
        if skip_existing.value and _has_report and _has_data:
            with out_run:
                print(f'⤼ [{file_id}] skipped (report + epoch data already exist).')
            progress.value = idx + 1
            continue
        if skip_existing.value and (_has_report != _has_data):
            with out_run:
                print(f'⚠ [{file_id}] '
                      + ('report exists but epoch data missing' if _has_report else 'epoch data exists but report missing')
                      + ' — reprocessing to restore consistency.')

        attempted_ids.add(file_id)

        try:  # [A] catch-all par participant — toute exception imprévue est capturée ici

            cdt_candidates = [f for f in curry_folder.rglob('*') if os.path.normcase(f.stem) == os.path.normcase(file_id) and f.suffix == '.cdt']
            if not cdt_candidates:
                with out_run:
                    print(f'[{file_id}] .cdt not found in {curry_folder}')
                failed.append({'file_id': file_id, 'reason': '.cdt not found'})
                progress.value = idx + 1
                continue
            cdt_path = cdt_candidates[0]

            cdt_rel   = cdt_path.parent.relative_to(curry_folder)
            deriv_dir = out_root / 'derivatives' / cdt_rel
            deriv_dir.mkdir(parents=True, exist_ok=True)

            if file_id not in config_dict:
                with out_run:
                    print(f'[{file_id}] Not found in JSON config — skipped.')
                failed.append({'file_id': file_id, 'reason': 'not in JSON config'})
                progress.value = idx + 1
                continue

            sub_config        = config_dict[file_id]
            ch_ui             = participant_ui[file_id]['channels']
            selected_channels = [ch for ch, cb in ch_ui.items() if cb.value]
            if not selected_channels:
                with out_run:
                    print(f'[{file_id}] No channels selected — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no channels selected'})
                progress.value = idx + 1
                continue

            set_phase('loading EDF…')
            # [load] Read the Curry header lazily (preload=False). All Curry channels share a
            # single sampling rate, so the EDF include=-at-read trick is unnecessary: we pick the
            # montage channels (original names from remap.keys()) then load only those from disk.
            try:
                raw = mne.io.read_raw_curry(str(cdt_path), preload=False, verbose='ERROR')
                _montage = [ch for ch in sub_config.get('remap', {}).keys() if ch in raw.ch_names]
                raw.pick(_montage)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error loading Curry file: {e}')
                failed.append({'file_id': file_id, 'reason': f'Curry loading: {e}'})
                progress.value = idx + 1
                continue

            # [B] Rename original channel names -> remapped names — non-fatal in itself, but the
            # selection just below depends on it (cf. the 'present' guard).
            try:
                raw.rename_channels({k: v for k, v in sub_config['remap'].items() if k in raw.ch_names})
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error renaming channels: {e} — channels not renamed.')

            # selected_channels carries the *remapped* names (UI / quality_summary), same
            # namespace as raw after renaming. We drop the deselected channels BEFORE
            # load_data() so only the kept channels are read from disk.
            present = [ch for ch in selected_channels if ch in raw.ch_names]
            if not present:
                with out_run:
                    print(f'[{file_id}] No selected channels found after renaming — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no channels after rename'})
                progress.value = idx + 1
                continue
            to_drop = [ch for ch in raw.ch_names if ch not in present]
            if to_drop:
                raw.drop_channels(to_drop)
            try:
                raw.load_data()   # reads only the kept channels from disk
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error reading Curry data: {e}')
                failed.append({'file_id': file_id, 'reason': f'Curry data loading: {e}'})
                progress.value = idx + 1
                continue

            progress_step.value += COST_LOAD   # Load complete
            # [C] Resampling — fatal
            if do_resample and target_freq is not None:
                set_phase('resampling…')
                try:
                    raw.resample(target_freq, npad='auto', verbose=False)
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Error during resampling: {e}')
                    failed.append({'file_id': file_id, 'reason': f'resampling: {e}'})
                    progress.value = idx + 1
                    continue
                progress_step.value += COST_RESAMPLE

            # [D] Re-référencement — non-fatal (warn + continuer sans re-ref)
            ref_channels = sub_config.get('ref_channels', [])
            try:
                if ref_channels == 'average':
                    raw.set_eeg_reference(ref_channels='average', verbose=False)
                elif isinstance(ref_channels, list) and ref_channels:
                    ref_present = [ch for ch in ref_channels if ch in raw.ch_names]
                    if ref_present:
                        raw.set_eeg_reference(ref_channels=ref_present, verbose=False)
                        raw.drop_channels(ref_present)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error during re-referencing: {e} — skipped.')

            # [D2] Notch filter — fatal
            if do_notch:
                set_phase('notch filtering…')
                try:
                    raw.notch_filter(freqs=notch_freq_val, verbose=False)
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Error during notch filtering: {e}')
                    failed.append({'file_id': file_id, 'reason': f'notch filtering: {e}'})
                    progress.value = idx + 1
                    continue
                progress_step.value += COST_NOTCH

            # [E] Filtrage — fatal
            if do_filter:
                set_phase('band-pass filtering…')
                try:
                    raw.filter(
                        l_freq=l_freq_val, h_freq=h_freq_val,
                        l_trans_bandwidth='auto', h_trans_bandwidth='auto',
                        filter_length='auto', method='fir', phase='zero-double',
                        fir_window='hamming', fir_design='firwin', verbose=False
                    )
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Error during filtering: {e}')
                    failed.append({'file_id': file_id, 'reason': f'filtering: {e}'})
                    progress.value = idx + 1
                    continue
                progress_step.value += COST_FILTER

            sf = raw.info['sfreq']

            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(os.path.normcase(_hypno_name), cdt_path.parent / _hypno_name)
            if not hypno_path.exists():
                with out_run:
                    print(f'[{file_id}] Hypnogram not found: {hypno_path.name}')
                failed.append({'file_id': file_id, 'reason': f'hypno not found: {hypno_path.name}'})
                progress.value = idx + 1
                continue
            try:
                expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error loading hypnogram: {e}')
                failed.append({'file_id': file_id, 'reason': f'hypno loading: {e}'})
                progress.value = idx + 1
                continue

            expected_epochs = int(np.floor(raw.n_times / sf / 30))
            n_hyp = len(expert_hypno)
            n_over = n_hyp - expected_epochs
            if 0 < n_over <= 1:
                # Recording ends mid-epoch: the last epoch is partial (< 30 s) but was still
                # scored. Drop the overhanging score and epoch only the complete 30 s epochs.
                last_sec = raw.n_times / sf - expected_epochs * 30
                expert_hypno = expert_hypno[:expected_epochs]
                with out_run:
                    print(f'[{file_id}] Last epoch {last_sec:.1f}s (< 30s) skipped; hypnogram '
                          f'trimmed {n_hyp} -> {expected_epochs} to match complete 30s epochs.')
            elif n_over != 0:
                with out_run:
                    print(f'[{file_id}] Hypnogram/EEG length mismatch ({n_hyp} '
                          f'vs {expected_epochs} epochs) — skipped.')
                failed.append({'file_id': file_id, 'reason': 'hypno/EEG length mismatch'})
                progress.value = idx + 1
                continue

            stage_mapping = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
            hypno_vec = np.full(len(expert_hypno), -1, dtype=float)
            for stage, value in stage_mapping.items():
                hypno_vec[expert_hypno == stage] = value

            # [F] Epoching — fatal
            set_phase('epoching…')
            try:
                epochs = mne.make_fixed_length_epochs(raw, duration=30, preload=True, verbose=False)
                epochs = epochs[:len(expert_hypno)]
                epochs.events[:, 2] = hypno_vec
                epochs.event_id     = stage_mapping
                n_epochs = len(epochs)
                del raw   # free raw._data (unused after epoching) — Curry per-channel memory
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error creating epochs: {e}')
                failed.append({'file_id': file_id, 'reason': f'epoching: {e}'})
                progress.value = idx + 1
                continue

            if n_epochs == 0:
                with out_run:
                    print(f'[{file_id}] No epochs — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no epochs'})
                progress.value = idx + 1
                continue

            progress_step.value += COST_EPOCH   # Epoching complete
            hypno_epochs   = expert_hypno[:n_epochs]

            set_phase('computing PSD / 1f…')
            # PSD — non-fatal (rejet 1/f désactivé si échec)
            try:
                n_per_seg = int(4 * sf)
                n_overlap = int(n_per_seg / 2)
                fmax_psd  = min(fit_fmax_val, sf / 2 - 0.5)  # PSD spans up to the 1/f fit max
                psds_obj  = epochs.compute_psd(
                    method='welch', fmin=0.5, fmax=fmax_psd,
                    n_fft=n_per_seg, n_overlap=n_overlap, n_per_seg=n_per_seg,
                    window='hann', verbose=False
                )
                psds_data_uV2 = psds_obj.get_data() * 1e12  # µV²/Hz
                psd_freqs     = psds_obj.freqs
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] PSD warning: {e} — 1/f rejection disabled.')
                psds_data_uV2 = None
                psd_freqs     = None

            # Event-based rejection mask (epoch-level) — non-fatal: a file with no event
            # companion just keeps the other 5 methods (never added to 'failed').
            event_epoch_mask = None
            event_type_masks = None   # {canonical type: (n_epochs,) bool mask} for the report breakdown
            if do_event:
                try:
                    events_df, ev_src = load_events(cdt_path, csv_suffix.value)
                    if events_df is None:
                        with out_run:
                            print(f'[{file_id}] ⚠ No event companion found — event-based rejection skipped for this file.')
                    else:
                        event_epoch_mask = compute_event_epoch_mask(
                            events_df, event_remap_dict, event_types, n_epochs, 30.0
                        )
                        event_type_masks = {
                            t: compute_event_epoch_mask(events_df, event_remap_dict, [t], n_epochs, 30.0)
                            for t in sorted(event_types)
                        }
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] ⚠ Event rejection error: {e} — skipped for this file.')
                    event_epoch_mask = None
                    event_type_masks = None

            _reject_base = progress_step.value   # Reject segment base (Curry callback fills within it)
            # Masques de rejet — fatal
            set_phase('rejecting epochs…')
            try:
                def _rej_progress(done, total, msg=''):
                    progress_step.value = _reject_base + int(COST_REJECT * done / max(1, total))
                    progress_step_label.value = f'{file_id} · rejecting · {msg} ({done}/{total})'
                mask_dict = compute_rejection_masks(
                    epochs, sf, hypno_epochs,
                    ptp_thresh, flat_thresh_val, grad_thresh_val,
                    psd_freqs, psds_data_uV2, thresh_error_val, thresh_r2_val,
                    event_epoch_mask=event_epoch_mask,
                    fit_fmin=fit_fmin_val, fit_fmax=fit_fmax_val, progress=_rej_progress
                )
                combined_matrix = build_combined_method_matrix(mask_dict)
                progress_step.value = _reject_base + COST_REJECT   # Rejection complete
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error computing rejection masks: {e}')
                failed.append({'file_id': file_id, 'reason': f'rejection masks: {e}'})
                progress.value = idx + 1
                continue

            reject_epoch = np.zeros(n_epochs, dtype=bool)
            for name in mask_dict:
                reject_epoch |= mask_dict[name].any(axis=1)

            method_priority = [m for m in ['amplitude', 'gradient', 'flat', '1f_error', '1f_r2', 'event']
                               if m in mask_dict]
            reject_method   = np.full(n_epochs, 'none', dtype=object)
            for name in reversed(method_priority):
                reject_method[mask_dict[name].any(axis=1)] = name
            n_methods = sum(mask_dict[name].any(axis=1).astype(int) for name in method_priority)
            reject_method[n_methods > 1] = 'multiple'

            meta_rows = []
            for ei in range(n_epochs):
                row = {
                    'epoch_idx':     ei,
                    'stage':         hypno_epochs[ei],
                    'reject_flag':   bool(reject_epoch[ei]),
                    'reject_method': reject_method[ei],
                }
                for name in method_priority:
                    col = 'flag_' + name.replace('/', '').replace(' ', '')
                    row[col] = bool(mask_dict[name][ei].any())
                meta_rows.append(row)
            metadata_df = pd.DataFrame(meta_rows)

            # [G] Sauvegarde .fif — fatal (produit principal)
            set_phase('saving epochs (.fif)…')
            try:
                epochs.metadata = metadata_df
                epochs.save(str(deriv_dir / f'{file_id}_all-epo.fif'), overwrite=True, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error saving .fif: {e}')
                failed.append({'file_id': file_id, 'reason': f'fif save: {e}'})
                progress.value = idx + 1
                continue

            progress_step.value = _reject_base + COST_REJECT + COST_SAVE   # .fif saved
            # [G] Sauvegarde des paramètres de preprocessing/rejet (sidecar JSON) — non-fatal.
            # Lu par l'outil 7 (QC des epochs rejetées) pour tracer les seuils / calculer les marges.
            try:
                methods_run = [m for m in METHOD_ORDER if m in mask_dict]
                preproc_params = {
                    'file_id': file_id,
                    'sfreq_hz': float(epochs.info['sfreq']),
                    'channels': list(epochs.ch_names),
                    'custom_stages': list(custom_stages),
                    'methods_run': methods_run,
                    'resample': {'applied': bool(do_resample),
                                 'target_freq_hz': float(target_freq) if do_resample else None},
                    'filter': {'applied': bool(do_filter),
                               'l_freq_hz': float(l_freq_val) if do_filter else None,
                               'h_freq_hz': float(h_freq_val) if do_filter else None},
                    'notch': {'applied': bool(do_notch),
                              'freq_hz': float(notch_freq_val) if do_notch else None},
                    'rejection_thresholds': {
                        'amplitude_ptp_uV': {s: float(v) for s, v in ptp_thresh.items()},
                        'flat_ptp_uV': float(flat_thresh_val),
                        'gradient_uV_per_sample': float(grad_thresh_val),
                        '1f_mae_max': float(thresh_error_val),
                        '1f_r2_min': float(thresh_r2_val),
                        '1f_fit_range_hz': [float(fit_fmin_val), float(fit_fmax_val)],
                    },
                    'event_rejection': {'applied': bool(do_event),
                                        'event_types': sorted(event_types) if do_event else []},
                }
                with open(str(deriv_dir / f'{file_id}_preprocessing_params.json'), 'w', encoding='utf-8') as _pf:
                    json.dump(preproc_params, _pf, indent=2, ensure_ascii=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error saving preprocessing_params.json: {e}')

            # [H] Context channels (EOG L/R, EMG, ECG) companion — optional, non-fatal.
            # Persist the declared context channels (from tool 2's remap_reref_persubject.json) as
            # {file_id}_context-epo.fif, epoched identically to the EEG (fixed 30 s from t=0, same
            # count) so tool 7 can stack them under the montage without reloading the raw EDF.
            try:
                ctx_map = sub_config.get('context_channels') or {}
                role_type  = {'eog_left': 'eog', 'eog_right': 'eog', 'emg': 'emg', 'ecg': 'ecg'}
                role_label = {'eog_left': 'EOG-L', 'eog_right': 'EOG-R', 'emg': 'EMG', 'ecg': 'ECG'}
                # original EDF channel name -> role; keep declared, non-null entries only
                orig_to_role = {}
                for _role in ('eog_left', 'eog_right', 'emg', 'ecg'):
                    _orig = ctx_map.get(_role)
                    if _orig:
                        orig_to_role[_orig] = _role
                if orig_to_role:
                    ctx_probe = mne.io.read_raw_curry(str(cdt_path), preload=False, verbose='ERROR')
                    present = [o for o in orig_to_role if o in ctx_probe.ch_names]
                    if present:
                        ctx_raw = ctx_probe.pick(present)
                        ctx_raw.load_data()
                        ctx_raw.rename_channels({o: role_label[orig_to_role[o]] for o in present})
                        ctx_raw.set_channel_types(
                            {role_label[orig_to_role[o]]: role_type[orig_to_role[o]] for o in present},
                            verbose=False)
                        if do_resample:
                            ctx_raw.resample(target_freq, npad='auto', verbose=False)
                        # Display-friendly filtering per role so the tool-7 epoch montage is readable
                        # (AASM-like): EOG band-pass 0.3–35 Hz, EMG high-pass 10 Hz, ECG band-pass
                        # 0.5–40 Hz. This is a *display* companion, so filtering it in place is fine.
                        _nyq = ctx_raw.info['sfreq'] / 2.0
                        _eog = [role_label[orig_to_role[o]] for o in present if orig_to_role[o] in ('eog_left', 'eog_right')]
                        _emg = [role_label[orig_to_role[o]] for o in present if orig_to_role[o] == 'emg']
                        _ecg = [role_label[orig_to_role[o]] for o in present if orig_to_role[o] == 'ecg']
                        if _eog:
                            ctx_raw.filter(0.3, min(35.0, _nyq - 0.5), picks=_eog, verbose=False)
                        if _emg:
                            ctx_raw.filter(10.0, None, picks=_emg, verbose=False)
                        if _ecg:
                            ctx_raw.filter(0.5, min(40.0, _nyq - 0.5), picks=_ecg, verbose=False)
                        ctx_epochs = mne.make_fixed_length_epochs(ctx_raw, duration=30, preload=True, verbose=False)
                        ctx_epochs.save(str(deriv_dir / f'{file_id}_context-epo.fif'), overwrite=True, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Could not save context channels companion: {e}')

            # [G] Sauvegarde epoch log TSV individuel — non-fatal
            try:
                epoch_log = build_epoch_rejection_log(mask_dict, hypno_epochs, file_id)
                epoch_log.to_csv(str(reports_dir / f'{file_id}_epoch_rejection.tsv'), sep='\t', index=False)
                all_epoch_logs.append(epoch_log)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error saving epoch_rejection.tsv: {e}')

            # [G] Sauvegarde rejection summary TSV individuel — non-fatal
            rej_summary = None
            try:
                rej_summary = build_rejection_summary(mask_dict, hypno_epochs, file_id, epochs.ch_names, custom_stages=custom_stages)
                rej_summary.to_csv(str(reports_dir / f'{file_id}_rejection_summary.tsv'), sep='\t', index=False)
                all_rejection_summaries.append(rej_summary)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error saving rejection_summary.tsv: {e}')

            n_rejected = int(reject_epoch.sum())
            pct_rej    = 100 * n_rejected / n_epochs if n_epochs > 0 else 0.0

            # [H] Rapport HTML + heatmap — non-fatal (.fif déjà sauvegardé)
            set_phase('building report…')
            try:
                report = mne.Report(title=f'{file_id} — Preprocessing Phase 2', verbose=False)

                fig_hm = plot_rejection_heatmap(combined_matrix, epochs.ch_names, hypno_epochs, file_id, custom_stages=custom_stages)
                report.add_figure(fig=fig_hm, title='Rejection heatmap', section='Artifacts', image_format='PNG')
                plt.close(fig_hm)

                summary_html = build_stage_method_html(rej_summary)
                report.add_html(
                    html=(f'<p><b>Total epochs:</b> {n_epochs} | '
                          f'<b>Rejected:</b> {n_rejected} ({pct_rej:.1f}%)</p>'
                          + summary_html),
                    title='Rejection summary', section='Artifacts'
                )

                # Per-channel × per-stage rejection breakdown (which channel drives the rejection
                # in which stage). Report-only; built from the in-memory per-(epoch, channel) masks.
                report.add_html(
                    html=build_channel_stage_html(mask_dict, hypno_epochs, epochs.ch_names,
                                                  custom_stages=custom_stages),
                    title='Rejection by channel × stage', section='Artifacts'
                )

                # Event-based rejection broken down per event type (helps decide which event
                # types reject too many epochs). Shown only when event rejection was enabled.
                if do_event:
                    event_type_html = build_event_type_html(event_type_masks, hypno_epochs,
                                                             custom_stages=custom_stages)
                    if event_type_html is None:
                        event_type_html = ('<p><em>Event-based rejection was enabled but no event '
                                           'companion was found for this file.</em></p>')
                    report.add_html(html=event_type_html,
                                    title='Event-based rejection by type', section='Artifacts')

                params_html = '<table style="border-collapse:collapse;font-size:.9em;">'
                if do_resample:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    f'<td style="padding:3px 10px;">{target_freq} Hz</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    '<td style="padding:3px 10px;">no</td></tr>')
                if do_filter:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Filter</b></td>'
                                    f'<td style="padding:3px 10px;">{l_freq_val}–{h_freq_val} Hz FIR zero-double Hamming</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Filter</b></td>'
                                    '<td style="padding:3px 10px;">no</td></tr>')
                if do_notch:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Notch</b></td>'
                                    f'<td style="padding:3px 10px;">{notch_freq_val} Hz FIR (MNE default)</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Notch</b></td>'
                                    '<td style="padding:3px 10px;">no</td></tr>')
                params_html += (
                    f'<tr><td style="padding:3px 10px;"><b>Amplitude thresholds</b></td>'
                    f'<td style="padding:3px 10px;">W={ptp_thresh["W"]} N1={ptp_thresh["N1"]} '
                    f'N2={ptp_thresh["N2"]} N3={ptp_thresh["N3"]} R={ptp_thresh["R"]} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Flat signal</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {flat_thresh_val} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Gradient</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {grad_thresh_val} µV/sample</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f error</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {thresh_error_val}</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f R²</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {thresh_r2_val}</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f fit range</b></td>'
                    f'<td style="padding:3px 10px;">{fit_fmin_val}–{fit_fmax_val} Hz</td></tr>'
                )
                if do_event:
                    _ev_state = 'applied' if event_epoch_mask is not None else 'no events found for this file'
                    params_html += (
                        f'<tr><td style="padding:3px 10px;"><b>Events</b></td>'
                        f'<td style="padding:3px 10px;">{_ev_state}: '
                        f'{", ".join(sorted(event_types))} '
                        f'(epoch containing event onset)</td></tr>'
                    )
                params_html += '</table>'
                report.add_html(html=params_html, title='Parameters', section='Artifacts')
                report.save(str(reports_dir / f'{file_id}_preprocessing_report.html'),
                            overwrite=True, open_browser=False, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error generating HTML report: {e}')

            cdt_rel_str = str(cdt_rel) if str(cdt_rel) != '.' else ''
            _deriv_rel = (f'derivatives/{cdt_rel_str}/{file_id}_all-epo.fif' if cdt_rel_str
                          else f'derivatives/{file_id}_all-epo.fif')
            with out_run:
                print(f'✓ [{file_id}]  {n_rejected}/{n_epochs} epochs rejected ({pct_rej:.1f}%)'
                      f'  →  {_deriv_rel}')

            progress_step.value = progress_step.max   # participant pipeline complete
            progress.value = idx + 1

        except Exception as e:  # [A] catch-all par participant
            with out_run:
                print(f'[{file_id}] UNEXPECTED ERROR: {repr(e)}')
            failed.append({'file_id': file_id, 'reason': f'unexpected: {repr(e)}'})
            progress.value = idx + 1

    # ---- Global outputs ---- [K]
    # Each global file is merged with existing data from previous runs.
    # Rows for attempted_ids are replaced (so a re-run on a participant replaces its old rows).
    # global_rejection_by_stage.tsv is regenerated from ALL individual _rejection_summary.tsv
    # files in reports_dir (mirrors quality_overview's dataset_overview.html strategy), so
    # it always reflects the complete state of the folder regardless of run history.

    # --- global_epoch_rejection.tsv ---
    # Regenerated from ALL individual *_epoch_rejection.tsv files present in reports_dir,
    # not just the current run (mirrors global_rejection_by_stage.tsv below). This makes the
    # global file interruption-safe: a participant processed in an earlier run and skipped in
    # this one still has its epoch rows re-included from disk instead of being silently dropped.
    global_epoch_path = reports_dir / 'global_epoch_rejection.tsv'
    try:
        # Exclude the global file itself — it shares the '_epoch_rejection.tsv' suffix.
        all_epoch_files = sorted(p for p in reports_dir.glob('*_epoch_rejection.tsv')
                                 if p.name != 'global_epoch_rejection.tsv')
        if all_epoch_files:
            all_epoch_loaded = []
            for epoch_path in all_epoch_files:
                try:
                    all_epoch_loaded.append(pd.read_csv(str(epoch_path), sep='\t', dtype={'file_id': str}))
                except Exception as _e:
                    with out_run:
                        print(f'⚠ Could not load {epoch_path.name}: {_e}')
            if all_epoch_loaded:
                df_epoch_to_save = pd.concat(all_epoch_loaded, ignore_index=True)
                df_epoch_to_save.to_csv(str(global_epoch_path), sep='\t', index=False)
                with out_run:
                    print('\nGlobal summary (per epoch)  →  global_epoch_rejection.tsv')
    except Exception as e:
        with out_run:
            print(f'⚠ Error saving global_epoch_rejection.tsv: {e}')

    # --- global_rejection_by_stage.tsv ---
    # Regenerated from all individual *_rejection_summary.tsv files present in reports_dir,
    # not just the current run — ensures the table always reflects the full dataset.
    global_stage_path = reports_dir / 'global_rejection_by_stage.tsv'
    try:
        all_summ_files = sorted(reports_dir.glob('*_rejection_summary.tsv'))
        if all_summ_files:
            all_summ_loaded = []
            for summ_path in all_summ_files:
                try:
                    all_summ_loaded.append(pd.read_csv(str(summ_path), sep='\t', dtype={'file_id': str}))
                except Exception as _e:
                    with out_run:
                        print(f'⚠ Could not load {summ_path.name}: {_e}')
            if all_summ_loaded:
                df_global_table = build_global_summary_table(all_summ_loaded, custom_stages=custom_stages)
                df_global_table.to_csv(str(global_stage_path), sep='\t', index=False)
                display_cols = ['stage'] + [c for c in df_global_table.columns if c.startswith('pct_')]
                pct_cols = [c for c in display_cols if c.startswith('pct_')]
                styled = (
                    df_global_table[display_cols].style
                    .set_table_styles([
                        {'selector': 'th', 'props': [('background', '#555'), ('color', '#fff'),
                                                      ('padding', '4px 12px'), ('text-align', 'left')]},
                        {'selector': 'td', 'props': [('padding', '3px 12px'),
                                                      ('border-bottom', '1px solid #eee')]},
                        {'selector': 'tr:nth-child(even) td', 'props': [('background', '#f8f8f8')]},
                    ])
                    .format({c: '{:.1f}' for c in pct_cols})
                    .hide(axis='index')
                )
                with out_run:
                    print('Global summary (per stage)  →  global_rejection_by_stage.tsv')
                    display(styled)
    except Exception as e:
        with out_run:
            print(f'⚠ Error generating global per-stage summary: {e}')

    # --- preprocessing_failed.tsv ---
    # Merge with existing: replace entries for attempted_ids (a participant that previously
    # failed and now succeeds is removed; one that fails again gets a fresh entry).
    failed_path = reports_dir / 'preprocessing_failed.tsv'
    if failed:
        try:
            df_failed_new = pd.DataFrame(failed)
            if failed_path.exists() and attempted_ids:
                df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
                df_failed_existing = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
                df_failed_merged = pd.concat([df_failed_existing, df_failed_new], ignore_index=True)
            else:
                df_failed_merged = df_failed_new
            df_failed_merged.to_csv(str(failed_path), sep='\t', index=False)
            with out_run:
                print(f'\n{len(failed)} participant(s) failed  →  preprocessing_failed.tsv')
        except Exception as e:
            with out_run:
                print(f'\n{len(failed)} participant(s) failed. '
                      f'⚠ Error saving preprocessing_failed.tsv: {e}')
    elif failed_path.exists() and attempted_ids:
        # All attempted participants succeeded: clean up any stale entries in failed.tsv.
        try:
            df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
            df_failed_cleaned = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
            if df_failed_cleaned.empty:
                failed_path.unlink()
            elif len(df_failed_cleaned) < len(df_failed_existing):
                df_failed_cleaned.to_csv(str(failed_path), sep='\t', index=False)
        except Exception as e:
            with out_run:
                print(f'⚠ Error updating preprocessing_failed.tsv: {e}')

    elapsed = time.time() - t_start
    mins, secs = divmod(elapsed, 60)
    elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'

    n_this_run = len(attempted_ids) - len(failed)
    n_total_in_folder = len(list(reports_dir.glob('*_preprocessing_report.html')))
    with out_run:
        print(f'\n=== Run complete ({elapsed_str}) ===')
        print(f'Processed this run    : {n_this_run}')
        if failed:
            print(f'Failed this run       : {len(failed)}')
        print(f'Total in folder       : {n_total_in_folder}')

    progress_lbl.value = f'Done. ({elapsed_str})'
    progress_step_label.value = ''
    btn.disabled = False


btn_run.on_click(run_preprocessing)

display(
    widgets.HBox([btn_load, load_info]),
    ctrl_row,
    vbox_participants,
    HTML('<hr style="margin:16px 0;">'),
    btn_run,
    widgets.HBox([progress, progress_lbl]),
    widgets.HBox([widgets.VBox([progress_step, step_legend], layout=widgets.Layout(margin='0')),
                  progress_step_label], layout=widgets.Layout(align_items='center')),
    out_run,
)